In [1]:
from pathlib import Path
from time import perf_counter
import json

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier

from src.utils.features import split_features

In [2]:
DATA_DIR = Path("../datasets/final/ml")
MODEL_DIR = Path("../models")
RESULTS_DIR = Path("../results/model_training")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 1
N_SPLITS = 5
N_JOBS = -1
SEARCH_STAGE = "search_01_initial"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [3]:
#configurações de treinamento / organização / origem dos dados
TRAINING_CONFIGS = {
    "cicids2017": ["cicids2017"],
    "unsw_nb15": ["unsw_nb15"],
    "iot23": ["iot23"],
    "cicids2017_unsw_nb15": ["cicids2017", "unsw_nb15"],
    "cicids2017_iot23": ["cicids2017", "iot23"],
    "unsw_nb15_iot23": ["unsw_nb15", "iot23"],
}

In [4]:
# atributos
reference_df = pd.read_parquet(DATA_DIR / "cicids2017_train.parquet")

numerical_features, categorical_features = split_features(reference_df)
feature_columns = numerical_features + categorical_features

del reference_df

print("features num:", len(numerical_features))
print("features cat:", len(categorical_features))
print("Total de features antes do one-hot:", len(feature_columns))

features num: 14
features cat: 1
Total de features antes do one-hot: 15


In [5]:
#pipeline pré-rocessamento
def build_pipeline(estimator):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", StandardScaler(), numerical_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"),categorical_features),
        ]
    )

    return Pipeline(
        steps=[ ("preprocessor", preprocessor),
                ("classifier", estimator),
        ]
    )

In [6]:
#modelos e espaços de busca
MODEL_CONFIGS = {
    "decision_tree": {
        "estimator": DecisionTreeClassifier(
            random_state=RANDOM_STATE,
        ),
        "param_grid": {
            "classifier__max_depth": [3, 5, 10, 20, None],
            "classifier__min_samples_leaf": [1, 5, 10],
        },
    },
    "random_forest": {
        "estimator": RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "param_grid": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [10, 20, None],
            "classifier__min_samples_leaf": [1, 5],
        },
    },
    "xgboost": {
        "estimator": XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "param_grid": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [3, 6],
            "classifier__learning_rate": [0.05, 0.1, 0.3],
        },
    },
    "linear_svm": {
        "estimator": LinearSVC(
            dual=False,
            max_iter=10000,
            random_state=RANDOM_STATE,
        ),
        "param_grid": {
            "classifier__C": [0.01, 0.1, 1.0, 10.0, 100.0],
        },
    },
    "mlp": {
        "estimator": MLPClassifier(
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
            random_state=RANDOM_STATE,
        ),
        "param_grid": {
            "classifier__hidden_layer_sizes": [
                (50,),
                (100,),
                (100, 50),
            ],
            "classifier__alpha": [0.0001, 0.001],
            "classifier__learning_rate_init": [0.0001, 0.001],
        },
    },
}

In [7]:
#modelos para rodar
MODELS_TO_RUN = [
    #"decision_tree",
    #"random_forest",
    #"xgboost",
    #"linear_svm",
    "mlp",
]

In [8]:
# kfold estratificado + métricas
# buscar: melhores hiperparâmetros através do f1-score
cross_validation = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = {
    "precision_attack": make_scorer(
        precision_score,
        pos_label=1,
        zero_division=0,
    ),
    "recall_attack": make_scorer(
        recall_score,
        pos_label=1,
        zero_division=0,
    ),
    "f1_attack": make_scorer(
        f1_score,
        pos_label=1,
        zero_division=0,
    ),
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
}

In [9]:
# carregamento de origem de treinamento
def load_training_data(dataset_names):
    columns = feature_columns + ["label_binary"]

    dataframes = []

    for dataset_name in dataset_names:
        df = pd.read_parquet( DATA_DIR / f"{dataset_name}_train.parquet", columns=columns)
        dataframes.append(df)

    training_df = pd.concat( dataframes, ignore_index=True)

    features = training_df[feature_columns]
    target = (training_df["label_binary"].eq("ATTACK").astype("int8"))

    return features, target

In [10]:
##treino, validação e save
def train_and_select_model(model_name, training_name, dataset_names):
    features, target = load_training_data(dataset_names)
    model_config = MODEL_CONFIGS[model_name]
    model_dir = MODEL_DIR / model_name
    results_dir = RESULTS_DIR / model_name

    model_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    search = GridSearchCV(
        estimator=build_pipeline(model_config["estimator"]),
        param_grid=model_config["param_grid"],
        scoring=scoring,
        refit="f1_attack",
        cv=cross_validation,
        n_jobs=N_JOBS,
        verbose=1,
        error_score="raise",
        return_train_score=False,
    )

    start = perf_counter()
    search.fit(features, target)
    elapsed_minutes = (perf_counter() - start) / 60

    cv_results = pd.DataFrame(search.cv_results_)

    candidate_columns = [
        "params",
        "mean_test_precision_attack",
        "std_test_precision_attack",
        "mean_test_recall_attack",
        "std_test_recall_attack",
        "mean_test_f1_attack",
        "std_test_f1_attack",
        "mean_test_accuracy",
        "std_test_accuracy",
        "mean_test_balanced_accuracy",
        "std_test_balanced_accuracy",
        "mean_fit_time",
        "rank_test_f1_attack",
    ]

    candidate_results = (
        cv_results[candidate_columns]
        .sort_values("rank_test_f1_attack")
    )
    candidate_results.insert(0, "random_state", RANDOM_STATE)
    candidate_results.insert(0, "n_splits", N_SPLITS)
    candidate_results.insert(0, "training_rows", len(features))
    candidate_results.insert(0, "training_source", training_name)
    candidate_results.insert(0, "search_stage", SEARCH_STAGE)
    candidate_results.insert(0, "model", model_name)

    print(
        f"\nResultados dos hiperparâmetros: "
        f"{model_name} - {training_name}"
    )
    display(candidate_results)

    candidate_results.to_csv(
        results_dir / f"{training_name}_{SEARCH_STAGE}.csv",
        index=False,
    )

    model_path = model_dir / f"{training_name}.joblib"

    joblib.dump(
        search.best_estimator_,
        model_path,
        compress=3,
    )

    best_result = cv_results.loc[search.best_index_]

    result = {
        "model": model_name,
        "search_stage": SEARCH_STAGE,
        "training_source": training_name,
        "training_rows": len(features),
        "best_parameters": json.dumps(
            search.best_params_,
            sort_keys=True,
        ),
        "precision_attack_mean": best_result["mean_test_precision_attack"],
        "precision_attack_std": best_result["std_test_precision_attack"],
        "recall_attack_mean": best_result["mean_test_recall_attack"],
        "recall_attack_std": best_result["std_test_recall_attack"],
        "f1_attack_mean": best_result["mean_test_f1_attack"],
        "f1_attack_std": best_result["std_test_f1_attack"],
        "accuracy_mean": best_result["mean_test_accuracy"],
        "accuracy_std": best_result["std_test_accuracy"],
        "balanced_accuracy_mean": best_result["mean_test_balanced_accuracy"],
        "balanced_accuracy_std": best_result["std_test_balanced_accuracy"],
        "elapsed_minutes": elapsed_minutes,
        "model_size_mb": (
                model_path.stat().st_size / 1024 ** 2
        ),
        "model_file": model_path.name,
    }

    return result

In [11]:
# exec
all_results = []

for model_name in MODELS_TO_RUN:
    model_results = []

    for training_name, dataset_names in TRAINING_CONFIGS.items():
        print(
            f"\nModelo: {model_name}"
            f"\nOrigem do treinamento: {training_name}"
        )

        result = train_and_select_model(
            model_name,
            training_name,
            dataset_names,
        )

        model_results.append(result)
        all_results.append(result)

    results_df = pd.DataFrame(model_results)
    results_df.to_csv(
        RESULTS_DIR / model_name / "summary.csv",
        index=False,
    )

all_results_df = pd.DataFrame(all_results)
display(all_results_df)


Modelo: mlp
Origem do treinamento: cicids2017
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Resultados dos hiperparâmetros: mlp - cicids2017


,model,search_stage,training_source,training_rows,n_splits,random_state,params,mean_test_precision_attack,std_test_precision_attack,mean_test_recall_attack,std_test_recall_attack,mean_test_f1_attack,std_test_f1_attack,mean_test_accuracy,std_test_accuracy,mean_test_balanced_accuracy,std_test_balanced_accuracy,mean_fit_time,rank_test_f1_attack
5,mlp,search_01_initial,cicids2017,235490,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.001}",0.978111,0.001851,0.991273,0.000807,0.984647,0.000919,0.993817,0.000375,0.992863,0.000410,46.180862,1
11,mlp,search_01_initial,cicids2017,235490,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.001}",0.977897,0.001278,0.991040,0.001197,0.984425,0.001161,0.993728,0.000468,0.992720,0.000729,31.890696,2
9,mlp,search_01_initial,cicids2017,235490,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.001}",0.977928,0.001316,0.983057,0.002377,0.980485,0.001659,0.992174,0.000661,0.988755,0.001289,31.263654,3
3,mlp,search_01_initial,cicids2017,235490,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.001}",0.977631,0.001012,0.982696,0.002410,0.980156,0.001473,0.992042,0.000585,0.988537,0.001255,23.253503,4
1,mlp,search_01_initial,cicids2017,235490,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (50,), 'classifier__learning_rate_init': 0.001}",0.976866,0.001295,0.982568,0.002756,0.979707,0.001616,0.991860,0.000640,0.988375,0.001408,25.373210,5
7,mlp,search_01_initial,cicids2017,235490,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (50,), 'classifier__learning_rate_init': 0.001}",0.976817,0.001378,0.982271,0.002469,0.979535,0.001550,0.991792,0.000616,0.988221,0.001282,23.874975,6
4,mlp,search_01_initial,cicids2017,235490,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.0001}",0.977042,0.001334,0.981273,0.001086,0.979153,0.001079,0.991643,0.000434,0.987754,0.000655,53.950129,7
10,mlp,search_01_initial,cicids2017,235490,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.0001}",0.976759,0.001172,0.980657,0.002415,0.978703,0.001557,0.991465,0.000618,0.987412,0.001275,45.079443,8
2,mlp,search_01_initial,cicids2017,235490,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.0001}",0.974370,0.001090,0.978258,0.001265,0.976309,0.000773,0.990505,0.000309,0.985912,0.000624,29.269524,9
8,mlp,search_01_initial,cicids2017,235490,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.0001}",0.974369,0.001071,0.978237,0.001287,0.976299,0.000809,0.990501,0.000323,0.985902,0.000645,28.593631,10



Modelo: mlp
Origem do treinamento: unsw_nb15
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Resultados dos hiperparâmetros: mlp - unsw_nb15


,model,search_stage,training_source,training_rows,n_splits,random_state,params,mean_test_precision_attack,std_test_precision_attack,mean_test_recall_attack,std_test_recall_attack,mean_test_f1_attack,std_test_f1_attack,mean_test_accuracy,std_test_accuracy,mean_test_balanced_accuracy,std_test_balanced_accuracy,mean_fit_time,rank_test_f1_attack
5,mlp,search_01_initial,unsw_nb15,235490,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.001}",0.938898,0.002037,0.996964,0.000428,0.967059,0.001068,0.986416,0.000456,0.990371,0.000330,39.391766,1
11,mlp,search_01_initial,unsw_nb15,235490,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.001}",0.938865,0.002411,0.996688,0.001043,0.966911,0.001372,0.986356,0.000581,0.990230,0.000617,33.607633,2
3,mlp,search_01_initial,unsw_nb15,235490,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.001}",0.939525,0.001821,0.993843,0.001460,0.965920,0.001525,0.985974,0.000632,0.988925,0.000916,40.605424,3
9,mlp,search_01_initial,unsw_nb15,235490,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.001}",0.939506,0.001838,0.993800,0.001660,0.965889,0.001319,0.985961,0.000547,0.988901,0.000886,38.455229,4
1,mlp,search_01_initial,unsw_nb15,235490,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (50,), 'classifier__learning_rate_init': 0.001}",0.938834,0.002065,0.994246,0.001335,0.965745,0.001512,0.985893,0.000631,0.989026,0.000843,36.243263,5
4,mlp,search_01_initial,unsw_nb15,235490,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.0001}",0.937900,0.002526,0.994925,0.000513,0.965570,0.001350,0.985808,0.000576,0.989227,0.000428,75.954697,6
10,mlp,search_01_initial,unsw_nb15,235490,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.0001}",0.937839,0.001952,0.994586,0.000996,0.965378,0.001213,0.985732,0.000510,0.989052,0.000605,62.863979,7
7,mlp,search_01_initial,unsw_nb15,235490,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (50,), 'classifier__learning_rate_init': 0.001}",0.939335,0.002161,0.992781,0.001186,0.965318,0.001436,0.985732,0.000601,0.988375,0.000746,30.971128,8
8,mlp,search_01_initial,unsw_nb15,235490,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.0001}",0.936480,0.001701,0.864792,0.007266,0.899191,0.003721,0.961226,0.001283,0.925063,0.003507,58.771095,9
2,mlp,search_01_initial,unsw_nb15,235490,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.0001}",0.937066,0.001784,0.863009,0.002547,0.898511,0.001571,0.961009,0.000578,0.924259,0.001261,50.611364,10



Modelo: mlp
Origem do treinamento: iot23
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Resultados dos hiperparâmetros: mlp - iot23


...


Modelo: mlp
Origem do treinamento: cicids2017_iot23
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Resultados dos hiperparâmetros: mlp - cicids2017_iot23


,model,search_stage,training_source,training_rows,n_splits,random_state,params,mean_test_precision_attack,std_test_precision_attack,mean_test_recall_attack,std_test_recall_attack,mean_test_f1_attack,std_test_f1_attack,mean_test_accuracy,std_test_accuracy,mean_test_balanced_accuracy,std_test_balanced_accuracy,mean_fit_time,rank_test_f1_attack
11,mlp,search_01_initial,cicids2017_iot23,470980,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.001}",0.987794,0.000863,0.987983,0.000751,0.987888,0.000651,0.995155,0.000261,0.992465,0.000420,60.251562,1
5,mlp,search_01_initial,cicids2017_iot23,470980,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.001}",0.987461,0.000528,0.988163,0.001666,0.987811,0.000791,0.995123,0.000313,0.992513,0.000812,67.890949,2
3,mlp,search_01_initial,cicids2017_iot23,470980,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.001}",0.977789,0.011224,0.982080,0.002596,0.979890,0.005293,0.991925,0.002175,0.988233,0.001528,56.852496,3
10,mlp,search_01_initial,cicids2017_iot23,470980,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.0001}",0.964615,0.000950,0.983704,0.001781,0.974066,0.001350,0.989524,0.000541,0.987342,0.001005,85.481420,4
4,mlp,search_01_initial,cicids2017_iot23,470980,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.0001}",0.964059,0.000852,0.983842,0.001831,0.973849,0.001170,0.989433,0.000467,0.987336,0.000966,86.247763,5
9,mlp,search_01_initial,cicids2017_iot23,470980,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.001}",0.964147,0.001041,0.983226,0.001194,0.973593,0.001106,0.989333,0.000446,0.987043,0.000725,38.776400,6
1,mlp,search_01_initial,cicids2017_iot23,470980,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (50,), 'classifier__learning_rate_init': 0.001}",0.962949,0.001033,0.981942,0.001668,0.972352,0.001088,0.988832,0.000435,0.986248,0.000874,35.091766,7
7,mlp,search_01_initial,cicids2017_iot23,470980,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (50,), 'classifier__learning_rate_init': 0.001}",0.963699,0.001395,0.981018,0.001106,0.972281,0.001055,0.988813,0.000428,0.985890,0.000645,30.950728,8
8,mlp,search_01_initial,cicids2017_iot23,470980,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.0001}",0.962074,0.001100,0.979702,0.000999,0.970808,0.001029,0.988216,0.000416,0.985023,0.000631,55.085919,9
2,mlp,search_01_initial,cicids2017_iot23,470980,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.0001}",0.961966,0.000996,0.979214,0.000870,0.970513,0.000781,0.988099,0.000316,0.984767,0.000498,48.556647,10



Modelo: mlp
Origem do treinamento: unsw_nb15_iot23
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Resultados dos hiperparâmetros: mlp - unsw_nb15_iot23


,model,search_stage,training_source,training_rows,n_splits,random_state,params,mean_test_precision_attack,std_test_precision_attack,mean_test_recall_attack,std_test_recall_attack,mean_test_f1_attack,std_test_f1_attack,mean_test_accuracy,std_test_accuracy,mean_test_balanced_accuracy,std_test_balanced_accuracy,mean_fit_time,rank_test_f1_attack
11,mlp,search_01_initial,unsw_nb15_iot23,470980,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.001}",0.966452,0.000823,0.990212,0.001747,0.978186,0.000591,0.991167,0.000231,0.990809,0.000785,82.137498,1
5,mlp,search_01_initial,unsw_nb15_iot23,470980,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.001}",0.966141,0.000605,0.989915,0.001657,0.977882,0.000655,0.991044,0.000257,0.990621,0.000772,87.888311,2
10,mlp,search_01_initial,unsw_nb15_iot23,470980,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.0001}",0.960686,0.005820,0.963098,0.018867,0.961759,0.008329,0.984717,0.003141,0.976610,0.008948,97.149196,3
4,mlp,search_01_initial,unsw_nb15_iot23,470980,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100, 50), 'classifier__learning_rate_init': 0.0001}",0.962871,0.000774,0.954977,0.021143,0.958789,0.010628,0.983630,0.004045,0.972885,0.010455,107.837830,4
7,mlp,search_01_initial,unsw_nb15_iot23,470980,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (50,), 'classifier__learning_rate_init': 0.001}",0.962618,0.001967,0.951474,0.018349,0.956933,0.009841,0.982908,0.003746,0.971120,0.009215,47.579763,5
3,mlp,search_01_initial,unsw_nb15_iot23,470980,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.001}",0.963357,0.002210,0.950210,0.018382,0.956652,0.009668,0.982815,0.003668,0.970588,0.009173,54.052149,6
9,mlp,search_01_initial,unsw_nb15_iot23,470980,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.001}",0.960033,0.006687,0.951378,0.018530,0.955565,0.009064,0.982337,0.003440,0.970727,0.008964,47.797709,7
1,mlp,search_01_initial,unsw_nb15_iot23,470980,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (50,), 'classifier__learning_rate_init': 0.001}",0.962423,0.001707,0.943703,0.017400,0.952895,0.009159,0.981373,0.003478,0.967247,0.008692,36.537945,8
2,mlp,search_01_initial,unsw_nb15_iot23,470980,5,1,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (100,), 'classifier__learning_rate_init': 0.0001}",0.960501,0.001150,0.925230,0.001803,0.942535,0.001373,0.977436,0.000529,0.957859,0.000996,74.533112,9
6,mlp,search_01_initial,unsw_nb15_iot23,470980,5,1,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_sizes': (50,), 'classifier__learning_rate_init': 0.0001}",0.960111,0.001325,0.925496,0.001664,0.942485,0.001191,0.977409,0.000462,0.957941,0.000878,66.269665,10


,model,search_stage,training_source,training_rows,best_parameters,precision_attack_mean,precision_attack_std,recall_attack_mean,recall_attack_std,f1_attack_mean,f1_attack_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,elapsed_minutes,model_size_mb,model_file
0,mlp,search_01_initial,cicids2017,235490,"{""classifier__alpha"": 0.0001, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.001}",0.978111,0.001851,0.991273,0.000807,0.984647,0.000919,0.993817,0.000375,0.992863,0.000410,3.323637,0.156008,cicids2017.joblib
1,mlp,search_01_initial,unsw_nb15,235490,"{""classifier__alpha"": 0.0001, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.001}",0.938898,0.002037,0.996964,0.000428,0.967059,0.001068,0.986416,0.000456,0.990371,0.000330,4.379708,0.158165,unsw_nb15.joblib
2,mlp,search_01_initial,iot23,235490,"{""classifier__alpha"": 0.0001, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.001}",0.999764,0.000142,0.991146,0.001361,0.995436,0.000704,0.998183,0.000279,0.995544,0.000684,1.579135,0.156344,iot23.joblib
3,mlp,search_01_initial,cicids2017_unsw_nb15,470980,"{""classifier__alpha"": 0.0001, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.001}",0.952751,0.001236,0.990021,0.002030,0.971027,0.000957,0.988184,0.000383,0.988873,0.000955,9.579976,0.156279,cicids2017_unsw_nb15.joblib
4,mlp,search_01_initial,cicids2017_iot23,470980,"{""classifier__alpha"": 0.001, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.001}",0.987794,0.000863,0.987983,0.000751,0.987888,0.000651,0.995155,0.000261,0.992465,0.000420,5.540747,0.151773,cicids2017_iot23.joblib
5,mlp,search_01_initial,unsw_nb15_iot23,470980,"{""classifier__alpha"": 0.001, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate_init"": 0.001}",0.966452,0.000823,0.990212,0.001747,0.978186,0.000591,0.991167,0.000231,0.990809,0.000785,7.239620,0.156699,unsw_nb15_iot23.joblib
